In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import re # limpieza del texto
import math # logaritmos
from collections import Counter, defaultdict # contar palabras

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/entrenamiento/entrenamiento.txt


In [2]:
# Se separan los labels y mensajes y estos se guardan en una lista de tuplas

data = []

with open("/kaggle/input/entrenamiento/entrenamiento.txt", encoding="utf-8") as file:
    for line in file:
        label, message = line.strip().split("\t", 1)
        data.append((label, message))

len(data)


5565

In [3]:
# Se van a separar los datos en 80/20
# Esto lo hacemos antes de crear el vocabulario para no tener data leakage

np.random.shuffle(data)

split = int(0.8 * len(data))
train_data = data[:split]
test_data = data[split:]

len(train_data), len(test_data)

(4452, 1113)

In [4]:
# Se realiza lo que es la limpieza de los datos, que es el pre procesamiento
# Se convierta todo a minúsculas
# Se elimina lo que es la puntuación y caracteres especiales

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-záéíóúñü\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


In [5]:
# Se realiza la tokenizacion 
# Con el concepto de Bag Words se ignora el orden y solo se cuentan las palbras

def tokenize(text):
    return clean_text(text).split()

In [6]:
# Se construye el vocabulario (V) usando train
# El vocabulario va a tener el conjunto de todas las palabras unicas vistas en el dataset

vocab = set()

for label, message in train_data:
    words = tokenize(message)
    vocab.update(words)

V = len(vocab)
V

7633

In [7]:
# Se separan lo que son los mensajes Spam y Ham

spam_messages = []
ham_messages = []

for label, message in train_data:
    if label == "spam":
        spam_messages.extend(tokenize(message))
    else:
        ham_messages.extend(tokenize(message))


In [8]:
# Se realiza el calculo de priors que representan la probabilidad inicial de que
# un mensaje pertenezca a cada clase antes de observar el texto

num_spam = sum(1 for label, _ in train_data if label == "spam")
num_ham = sum(1 for label, _ in train_data if label == "ham")
total = len(train_data)

P_spam = num_spam / total
P_ham = num_ham / total

P_spam, P_ham


(0.13701707097933513, 0.8629829290206649)

In [9]:
# Se realiza el conteo de palabras por clase
# Se cuentan cuantas veces aparece cada palabra dentro de 
# los mensajes de spam y ham por separado, lo que implementa
# la idea del Bag Words

spam_counts = Counter(spam_messages)
ham_counts = Counter(ham_messages)

total_spam_words = sum(spam_counts.values())
total_ham_words = sum(ham_counts.values())


In [10]:
# Se realiza el calculo de Likehoods con Laplace Smoothing
# Se calcula para cada palabra del vocabulario
# Usamos Laplace Smoothing con k = 1 para evitar probabilidades cero,
# lo cual es importante ya que una sola palabra con probabilidad cero
# haría que toda la probabilidad posterior sea cero

k = 1

likelihood_spam = {}
likelihood_ham = {}

for word in vocab:
    likelihood_spam[word] = (spam_counts[word] + k) / (total_spam_words + k * V)
    likelihood_ham[word] = (ham_counts[word] + k) / (total_ham_words + k * V)


In [11]:
# Se realiza la funcion de prediccion, que clasifica 
# un mensaje nuevo aplicando la regla de bayes

def predict(message):
    words = tokenize(message)

    log_spam = math.log(P_spam)
    log_ham = math.log(P_ham)

    for word in words:
        if word in vocab:
            log_spam += math.log(likelihood_spam[word])
            log_ham += math.log(likelihood_ham[word])

    return "spam" if log_spam > log_ham else "ham"


In [12]:
# Se aplica el modelo entrenado a cada mensaje del conjunto de prueba
# Para evaluar el desempeño del clasificador

y_true = []
y_pred = []

for label, message in test_data:
    y_true.append(label)
    y_pred.append(predict(message))


In [13]:
# Se evalua el desempeño del modelo usando una matriz de confusion
# para poder observar aciertos y errores que diferencian entre Spam y Ham,
# tambien se usa Acuarracy para tener el procentaje de mensajes correctamente
# clasificados

from sklearn.metrics import confusion_matrix, accuracy_score

cm = confusion_matrix(y_true, y_pred, labels=["spam", "ham"])
acc = accuracy_score(y_true, y_pred)

cm, acc

pd.DataFrame(
    cm,
    index=["Actual Spam", "Actual Ham"],
    columns=["Pred Spam", "Pred Ham"]
)


,Pred Spam,Pred Ham
Actual Spam,129,8
Actual Ham,6,970
